# ParentPaper: Colab-only runner

This notebook writes the full Python script on Colab and runs it against data in Google Drive at `DS340W/Combined`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, textwrap
BASE = '/content/drive/MyDrive/DS340W'
DATA = f'{BASE}/Combined'
OUT = f'{BASE}/parent_paper_predictions.csv'
SPLIT_OUT = BASE
print('BASE:', BASE)
print('DATA:', DATA)
print('DATA exists:', os.path.isdir(DATA))
print('Files in DATA (first 10):', os.listdir(DATA)[:10] if os.path.isdir(DATA) else 'missing')

BASE: /content/drive/MyDrive/DS340W
DATA: /content/drive/MyDrive/DS340W/Combined
DATA exists: True
Files in DATA (first 10): ['Crime_Enriched_PointLevel.csv', 'Crime_Enriched_PrecinctDaily.csv', 'Crime_Replication_PrecinctDaily.csv']


In [3]:
code = textwrap.dedent('''
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))


def build_knn_adjacency(coords: np.ndarray, k: int = 4) -> np.ndarray:
    n = coords.shape[0]
    adj = np.zeros((n, n), dtype=np.float32)
    dists = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1))
    for i in range(n):
        nn_idx = np.argsort(dists[i])[1 : k + 1]
        adj[i, nn_idx] = 1.0
    adj = np.maximum(adj, adj.T)
    np.fill_diagonal(adj, 1.0)
    return adj


def normalize_adj(adj: np.ndarray) -> np.ndarray:
    d = np.sum(adj, axis=1)
    d_inv_sqrt = np.power(d, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    D_inv_sqrt = np.diag(d_inv_sqrt)
    return D_inv_sqrt @ adj @ D_inv_sqrt


class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.lin = nn.Linear(in_feats, out_feats)

    def forward(self, x, adj_norm):
        h = torch.matmul(adj_norm, x)
        return self.lin(h)


class STGCNBlock(nn.Module):
    def __init__(self, in_feats, hidden_feats):
        super().__init__()
        self.gcn = GraphConv(in_feats, hidden_feats)
        self.act = nn.ReLU()

    def forward(self, x, adj_norm):
        return self.act(self.gcn(x, adj_norm))


class ProbSparseAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1, factor=5):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.factor = factor
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        sparsity = scores.max(dim=-1).values - scores.mean(dim=-1)

        u = min(t, max(1, int(self.factor * np.log(t + 1))))
        top_idx = torch.topk(sparsity, k=u, dim=-1).indices

        v_mean = v.mean(dim=-2, keepdim=True)
        context = v_mean.repeat(1, 1, t, 1)

        top_scores = scores.gather(-2, top_idx.unsqueeze(-1).repeat(1, 1, 1, t))
        top_attn = torch.softmax(top_scores, dim=-1)
        top_ctx = torch.matmul(top_attn, v)

        context.scatter_(-2, top_idx.unsqueeze(-1).repeat(1, 1, 1, self.head_dim), top_ctx)
        context = context.transpose(1, 2).contiguous().view(b, t, self.d_model)
        return self.out_proj(self.drop(context))


class InformerEncoderLayer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = ProbSparseAttention(d_model, n_heads=n_heads, dropout=dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x


class InformerEncoder(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [InformerEncoderLayer(d_model, n_heads, dropout) for _ in range(n_layers)]
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class HybridInformerSTGCN(nn.Module):
    def __init__(self, node_feat_dim, time_feat_dim, stgcn_hidden=128, d_model=64):
        super().__init__()
        self.stgcn = STGCNBlock(node_feat_dim, stgcn_hidden)
        self.temporal_proj = nn.Linear(time_feat_dim, d_model)
        self.informer = InformerEncoder(d_model=d_model, n_heads=4, n_layers=4)
        self.fuse = nn.Linear(stgcn_hidden + d_model, 1)

    def forward(self, spatial_x, temporal_x, adj_norm):
        b, n, t, f = temporal_x.shape
        spatial_h = self.stgcn(spatial_x, adj_norm)
        temporal_x = temporal_x.reshape(b * n, t, f)
        temporal_x = self.temporal_proj(temporal_x)
        temporal_h = self.informer(temporal_x)[:, -1, :]
        temporal_h = temporal_h.reshape(b, n, -1)
        fused = torch.cat([spatial_h, temporal_h], dim=-1)
        return self.fuse(fused).squeeze(-1)


def load_replication(path: Path, start_date: str, end_date: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['crime_date'] = pd.to_datetime(df['crime_date']).dt.date
    if 'precinct' not in df.columns:
        raise ValueError("Expected 'precinct' column in replication dataset.")
    if 'crime_type' not in df.columns:
        raise ValueError("Expected 'crime_type' column in replication dataset.")
    if 'crime_count' not in df.columns:
        raise ValueError("Expected 'crime_count' column in replication dataset.")
    s = pd.to_datetime(start_date).date()
    e = pd.to_datetime(end_date).date()
    df = df[(df['crime_date'] >= s) & (df['crime_date'] <= e)]
    return df


def ensure_features(df: pd.DataFrame) -> pd.DataFrame:
    dts = pd.to_datetime(df['crime_date'])
    if 'weekend' not in df.columns:
        df['weekend'] = (dts.dt.weekday >= 5).astype(int)
    if 'holiday' not in df.columns:
        df['holiday'] = 0

    if 'month' not in df.columns:
        df['month'] = dts.dt.month

    if 'e_mean' not in df.columns and 't_mean' in df.columns and 'rh_mean' in df.columns:
        T = df['t_mean'].astype(float)
        RH = df['rh_mean'].astype(float)
        df['e_mean'] = (RH / 100.0) * 6.105 * np.exp((17.2 * T) / (237.7 + T))

    if 'weekday_avg' not in df.columns or 'weekend_avg' not in df.columns:
        grp = df.groupby(['precinct', 'crime_type'])
        if 'weekday_avg' not in df.columns:
            df['weekday_avg'] = grp['crime_count'].transform(lambda s: s[df.loc[s.index, 'weekend'] == 0].mean())
        if 'weekend_avg' not in df.columns:
            df['weekend_avg'] = grp['crime_count'].transform(lambda s: s[df.loc[s.index, 'weekend'] == 1].mean())

    if 'month_avg' not in df.columns:
        df['month_avg'] = df.groupby(['precinct', 'crime_type', 'month'])['crime_count'].transform('mean')

    if 'count_avg' not in df.columns:
        df = df.sort_values(['precinct', 'crime_type', 'crime_date'])
        df['count_avg'] = df.groupby(['precinct', 'crime_type'])['crime_count'].shift(1).fillna(0)

    return df


def complete_date_grid(df: pd.DataFrame) -> pd.DataFrame:
    dates = pd.date_range(df['crime_date'].min(), df['crime_date'].max(), freq='D').date
    precincts = sorted(df['precinct'].unique())
    crime_types = sorted(df['crime_type'].unique())
    grid = pd.MultiIndex.from_product(
        [dates, precincts, crime_types],
        names=['crime_date', 'precinct', 'crime_type'],
    )
    df = df.set_index(['crime_date', 'precinct', 'crime_type']).reindex(grid).reset_index()
    df['crime_count'] = df['crime_count'].fillna(0)
    return df


def fill_daily_features(df: pd.DataFrame) -> pd.DataFrame:
    daily_cols = [
        't_mean', 't_min', 't_max', 'rh_mean', 'wind_mean',
        'at_mean', 'tw_mean', 'di_mean', 'obs_count',
        'weekend', 'holiday', 'is_christian_holiday', 'is_islamic_holiday',
        'is_jewish_holiday', 'is_hindu_holiday', 'e_mean', 'month'
    ]
    present = [c for c in daily_cols if c in df.columns]
    if not present:
        return df
    daily = df.groupby('crime_date')[present].first().reset_index()
    df = df.drop(columns=present).merge(daily, on='crime_date', how='left')
    return df


def build_series(df: pd.DataFrame, precincts, lookback, feature_cols):
    dates = sorted(df['crime_date'].unique())
    n_nodes = len(precincts)
    n_features = len(feature_cols)
    series = np.zeros((len(dates), n_nodes, n_features), dtype=np.float32)

    for i, day in enumerate(dates):
        day_df = df[df['crime_date'] == day].set_index('precinct').reindex(precincts)
        series[i] = day_df[feature_cols].values.astype(np.float32)

    X, y, target_dates = [], [], []
    for i in range(lookback, len(dates)):
        X.append(series[i - lookback : i])
        y.append(series[i, :, 0])
        target_dates.append(dates[i])
    X = np.stack(X) if X else np.zeros((0, lookback, n_nodes, n_features), dtype=np.float32)
    y = np.stack(y) if y else np.zeros((0, n_nodes), dtype=np.float32)
    if X.size:
        X = np.transpose(X, (0, 2, 1, 3))
    else:
        X = np.zeros((0, n_nodes, lookback, n_features), dtype=np.float32)
    return X, y, target_dates


def build_spatial_features(df: pd.DataFrame, precincts):
    dates = sorted(df['crime_date'].unique())
    idx = {d: i for i, d in enumerate(dates)}
    series = np.zeros((len(dates), len(precincts)), dtype=np.float32)
    for i, day in enumerate(dates):
        day_df = df[df['crime_date'] == day].set_index('precinct').reindex(precincts)
        series[i] = day_df['crime_count'].values.astype(np.float32)

    def get_at(day, offset):
        j = idx[day] - offset
        if j < 0:
            return np.zeros(len(precincts), dtype=np.float32)
        return series[j]

    spatial_X = []
    for day in dates:
        prox = np.stack([get_at(day, 1), get_at(day, 2), get_at(day, 3)], axis=-1)
        peri = np.stack([get_at(day, 7), get_at(day, 14), get_at(day, 21)], axis=-1)
        trend = get_at(day, 365 * 3)[:, None]
        spatial_X.append(np.concatenate([prox, peri, trend], axis=-1))
    return np.stack(spatial_X), dates


def build_adjacency(point_path: Path | None, precincts, k=4):
    if point_path and point_path.exists():
        cdf = pd.read_csv(point_path, usecols=['addr_pct_cd', 'latitude', 'longitude'], low_memory=False)
        cdf = cdf.dropna(subset=['addr_pct_cd', 'latitude', 'longitude'])
        cdf['precinct'] = cdf['addr_pct_cd'].astype(str)
        cent = cdf.groupby('precinct')[['latitude', 'longitude']].mean().reindex(precincts)
        coords = np.nan_to_num(cent.values, nan=0.0)
        adj = build_knn_adjacency(coords, k=k)
    else:
        n = len(precincts)
        adj = np.eye(n, dtype=np.float32)
        for i in range(n - 1):
            adj[i, i + 1] = 1.0
            adj[i + 1, i] = 1.0
    return normalize_adj(adj)


def train_model(X, y, spatial_X, adj_norm, epochs=200, batch=32, lr=0.01, patience=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.float32, device=device)
    spatial_t = torch.tensor(spatial_X, dtype=torch.float32, device=device)
    adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)

    node_feat_dim = spatial_t.shape[-1]
    time_feat_dim = X_t.shape[-1]
    model = HybridInformerSTGCN(node_feat_dim=node_feat_dim, time_feat_dim=time_feat_dim).to(device)
    opt = torch.optim.NAdam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best = float('inf')
    patience_left = patience

    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(X_t.shape[0])
        epoch_loss = 0.0

        for i in range(0, X_t.shape[0], batch):
            idx = perm[i : i + batch]
            xb = X_t[idx]
            yb = y_t[idx]
            sb = spatial_t[idx]

            opt.zero_grad()
            pred = model(sb, xb, adj_t)
            loss = rmse(pred, yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()

        scheduler.step()

        if epoch_loss < best:
            best = epoch_loss
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    return model, device


def eval_model(model, device, X, y, spatial_X, adj_norm):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32, device=device)
        y_t = torch.tensor(y, dtype=torch.float32, device=device)
        spatial_t = torch.tensor(spatial_X, dtype=torch.float32, device=device)
        adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)
        pred = model(spatial_t, X_t, adj_t)
        mae = torch.mean(torch.abs(pred - y_t)).item()
        rmse_v = rmse(pred, y_t).item()
    return pred.cpu().numpy(), mae, rmse_v


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--replication', default='Datasets/Clean/Combined/Crime_Replication_PrecinctDaily.csv')
    ap.add_argument('--point', default='Datasets/Clean/Combined/Crime_Enriched_PointLevel.csv')
    ap.add_argument('--start', default='2015-01-01')
    ap.add_argument('--end', default='2020-03-10')
    ap.add_argument('--train_start', default='2015-01-01')
    ap.add_argument('--train_end', default='2019-12-31')
    ap.add_argument('--test_start', default='2020-01-01')
    ap.add_argument('--test_end', default='2020-01-07')
    ap.add_argument('--lookback', type=int, default=8)
    ap.add_argument('--epochs', type=int, default=200)
    ap.add_argument('--batch', type=int, default=32)
    ap.add_argument('--lr', type=float, default=0.01)
    ap.add_argument('--patience', type=int, default=100)
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--k', type=int, default=4)
    ap.add_argument('--out', default='Datasets/Clean/parent_paper_predictions.csv')
    ap.add_argument('--split_out', default='Datasets/Clean')
    args = ap.parse_args()

    set_seed(args.seed)

    train_start = pd.to_datetime(args.train_start).date()
    train_end = pd.to_datetime(args.train_end).date()
    test_start = pd.to_datetime(args.test_start).date()
    test_end = pd.to_datetime(args.test_end).date()

    if not (train_start <= train_end):
        raise ValueError('train_start must be <= train_end.')
    if not (test_start <= test_end):
        raise ValueError('test_start must be <= test_end.')
    if not (train_end < test_start):
        raise ValueError('train_end must be before test_start to avoid leakage.')

    df = load_replication(Path(args.replication), args.start, args.end)
    df = ensure_features(df)
    df = complete_date_grid(df)
    df = fill_daily_features(df)

    precincts = sorted(df['precinct'].unique())
    adj_norm = build_adjacency(Path(args.point), precincts, k=args.k)

    train_dates = {d for d in df['crime_date'].unique() if train_start <= d <= train_end}
    test_dates = {d for d in df['crime_date'].unique() if test_start <= d <= test_end}

    parent_train = df[df['crime_date'].isin(train_dates)]
    parent_test = df[df['crime_date'].isin(test_dates)]

    split_dir = Path(args.split_out)
    split_dir.mkdir(parents=True, exist_ok=True)
    parent_train.to_csv(split_dir / 'parent_train.csv', index=False)
    parent_test.to_csv(split_dir / 'parent_test.csv', index=False)

    out_rows = []

    for crime_type in sorted(df['crime_type'].unique()):
        sub = df[df['crime_type'] == crime_type].copy()

        base_features = [
            'crime_count',
            'weekend',
            'holiday',
            'weekday_avg',
            'weekend_avg',
            'month_avg',
            'count_avg',
            't_mean',
            'rh_mean',
            'wind_mean',
            'e_mean',
            'U',
            'Ff',
        ]
        seen = set()
        feature_cols = []
        for c in base_features:
            if c in sub.columns and c not in seen:
                feature_cols.append(c)
                seen.add(c)

        X_all, y_all, target_dates = build_series(sub, precincts, args.lookback, feature_cols)
        spatial_all, spatial_dates = build_spatial_features(sub, precincts)
        spatial_idx = {d: i for i, d in enumerate(spatial_dates)}
        spatial_X_all = np.stack([spatial_all[spatial_idx[d]] for d in target_dates]) if target_dates else np.zeros((0, len(precincts), 7), dtype=np.float32)

        train_idx = [i for i, d in enumerate(target_dates) if d in train_dates]
        test_idx = [i for i, d in enumerate(target_dates) if d in test_dates]

        X_train = X_all[train_idx]
        y_train = y_all[train_idx]
        spatial_train = spatial_X_all[train_idx]

        X_test = X_all[test_idx]
        y_test = y_all[test_idx]
        spatial_test = spatial_X_all[test_idx]
        test_target_dates = [target_dates[i] for i in test_idx]

        model, device = train_model(
            X_train, y_train, spatial_train, adj_norm,
            epochs=args.epochs, batch=args.batch, lr=args.lr, patience=args.patience
        )
        preds, mae, rmse_v = eval_model(model, device, X_test, y_test, spatial_test, adj_norm)

        for i, day in enumerate(test_target_dates):
            for j, precinct in enumerate(precincts):
                out_rows.append({
                    'crime_type': crime_type,
                    'crime_date': day,
                    'precinct': precinct,
                    'y_true': float(y_test[i, j]),
                    'y_pred': float(preds[i, j]),
                })

        print(f'{crime_type}: MAE={mae:.4f} RMSE={rmse_v:.4f}')

    out_df = pd.DataFrame(out_rows)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(args.out, index=False)
    print(f'Wrote predictions to {args.out}')


if __name__ == '__main__':
    main()
''')
with open('/content/parentpaper_colab.py', 'w', encoding='utf-8') as f:
    f.write(code)
print('Wrote /content/parentpaper_colab.py')

Wrote /content/parentpaper_colab.py


In [4]:
!python /content/parentpaper_colab.py \
  --replication {DATA}/Crime_Replication_PrecinctDaily.csv \
  --point {DATA}/Crime_Enriched_PointLevel.csv \
  --out {OUT} \
  --split_out {SPLIT_OUT}

ASSAULT: MAE=2.4371 RMSE=3.2447
CRIMINAL_DAMAGE: MAE=1.5732 RMSE=3.7250
ROBBERY: MAE=0.6133 RMSE=0.8111
THEFT: MAE=2.0108 RMSE=2.5241
Wrote predictions to /content/drive/MyDrive/DS340W/parent_paper_predictions.csv
